# W7 deck figures — what changed for the final talk

The W6 deck's figures are built by [`15_w6_visuals.ipynb`](15_w6_visuals.ipynb) and are unchanged.
This notebook holds only the figures **revised or added for W7**, so the W6 notebook stays a record
of what was presented in week 6 rather than being edited under it.

**One revision so far.** Beat 3's Tier-1 tiles reported three bare means — 42% / 52% / 73% — with no
indication of how much a single region-season varies around them. The tiles now carry the middle
half of their own per-cell distribution beneath each headline. The prompt for this was the W7
confidence work in [`06_analysis.ipynb`](06_analysis.ipynb): once a per-cell confidence *ranking*
exists, presenting the mean alone understates what is known and overstates what one forecast is
worth.

Figures are written to `../img/` and consumed by `coursework/W7/MSDS696_W7_Deck.pptx` via
`coursework/W7/final_script.md`, which is the authoritative text.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image

sys.path.insert(0, "../src")
import w6_visuals as wv
from config import ProjectConfig
from panel import RegionSeasonPanel

cfg = ProjectConfig()
IMG = Path("../img")
IMG.mkdir(exist_ok=True)

# Same panel object the analysis notebooks use, so a figure cannot drift from the
# numbers 06 reports.
panel = RegionSeasonPanel.load(cfg)
print(f"{panel.rsc.region.nunique()} regions x {panel.rsc.season.nunique()} seasons "
      f"| test split at season_year >= {cfg.test_start} | shares window k={cfg.shares_k}")

## Beat 3 — the Tier-1 tiles, now carrying their spread

Three predictors of a region-season's next cause mix, scored on the held-out tail: the national
average mix, an even split, and the region's own k=7 trailing history. The tile value is
`1 - TVD`, which on a simplex is exactly the overlap between the predicted and actual composition —
so "share of the burned-acre composition placed on the right cause" is literal.

**What is new is the span beneath each bar: the middle half (p25–p75) of the per-cell
distribution.** The headline is an average over 3,949 held-out region-seasons; a planner receives
one of them. A bare mean invites the reading that every region-season lands near 73%, and they do
not.

**The span is acre-weighted, matching the headline.** Unweighted percentiles were tried first and
produced a figure that reads as a mistake — the national tile came out at a 42% headline against a
50–79% range, because that predictor fails hardest on exactly the big-burn cells the acre weighting
is dominated by, while doing adequately on the many small ones. Both numbers were correct and the
pair was unreadable. One denominator throughout is the same rule the spoken script applies to every
share it quotes.

**It is a spread across region-seasons, not a confidence interval on the mean.** The figure's drawn
label says "typical range" for that reason. No inferential claim is being made, and the notes forbid
calling it one.

In [ ]:
tiles = wv.plot_tier1_tiles(panel, IMG / "w6_tier1_tiles.png", k=cfg.shares_k)

for key in ("national", "naive", "history"):
    print(f"  {key:9s} {tiles[f'{key}_correct']:.1%}   "
          f"typical range {tiles[f'{key}_p25']:.0%}-{tiles[f'{key}_p75']:.0%}   "
          f"median {tiles[f'{key}_p50']:.0%}")
print(f"\n  {tiles['n_cells']:,} held-out region-seasons, season_year >= {tiles['test_start']}, "
      f"k={tiles['k']}")

Image(filename=tiles["out_path"], width=980)

**Finding.** The comparison survives the added honesty, and is sharper for it. The region's own
history spans **58%–91%** where the national average mix spans **21%–61%** — the two barely overlap,
so history is not merely better on average, it is better across most of the distribution.

The national tile is the one the spread most changes. Its headline of 42% is not a middling result
evenly spread; it reaches down to **21%** at the first quartile. That is beat 2's bimodality
reappearing as forecast error: a national average describes almost no individual region, so applying
it to a specific region-season fails badly and often.

**What the span does not say.** It is variation across region-seasons, not uncertainty about the
mean, and it does not identify *which* cells land at the bottom. That question is answered
separately in [`06_analysis.ipynb`](06_analysis.ipynb): a cell's own pre-season history dispersion
predicts its error (Spearman +0.484, 33 SD above a shuffled control), so the low end of this span is
substantially anticipated rather than random.